In [ ]:
# confirm connection with Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("warehouse")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    # pinned so the executors have a fixed address to call back on
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    # the worker has 2 cores total; without a cap this session holds them all
    # and any other notebook waits forever for an executor
    .config("spark.cores.max", 2)
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

# smoke test: round-trip a tiny DataFrame through the cluster
spark.range(5).selectExpr("id", "id * id as squared").show()


In [5]:
# create data warehouse
from pathlib import Path

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DateType,
)

WAREHOUSE = "/opt/data/warehouse"

# stage_trips — raw columns after header normalization, all string
STAGE_TRIPS_SCHEMA = StructType(
    [
        StructField("trip_id", StringType()),
        StructField("trip_duration", StringType()),
        StructField("start_station_id", StringType()),
        StructField("start_time", StringType()),
        StructField("start_station_name", StringType()),
        StructField("end_station_id", StringType()),
        StructField("end_time", StringType()),
        StructField("end_station_name", StringType()),
        StructField("bike_id", StringType()),
        StructField("user_type", StringType()),
        StructField("bike_model", StringType()),  # 2024+ only, null otherwise
        StructField("source_file", StringType()),
        StructField("source_year", IntegerType()),  # partition key
    ]
)

DIM_STATION_SCHEMA = StructType(
    [
        StructField("station_id", IntegerType()),
        StructField("station_name", StringType()),
    ]
)

DIM_USER_SCHEMA = StructType(
    [
        StructField("user_type_id", IntegerType()),
        StructField("user_type", StringType()),
    ]
)

FACT_TRIPS_SCHEMA = StructType(
    [
        StructField("trip_id", LongType()),
        StructField("trip_duration", IntegerType()),
        StructField("start_year", IntegerType()),  # partition key
        StructField("start_month", IntegerType()),  # partition key
        StructField("start_date", DateType()),
        StructField("start_day_of_week", IntegerType()),
        StructField("start_hour", IntegerType()),
        StructField("start_quarter", IntegerType()),
        StructField("end_year", IntegerType()),
        StructField("end_month", IntegerType()),
        StructField("end_date", DateType()),
        StructField("end_day_of_week", IntegerType()),
        StructField("end_hour", IntegerType()),
        StructField("end_quarter", IntegerType()),
        StructField("start_station_id", IntegerType()),
        StructField("end_station_id", IntegerType()),
        StructField("user_type_id", IntegerType()),
    ]
)

# table -> (schema, partition columns); paths are created by the ETL writes
TABLES = {
    "stage_trips": (STAGE_TRIPS_SCHEMA, ["source_year"]),
    "dim_station": (DIM_STATION_SCHEMA, []),
    "dim_user": (DIM_USER_SCHEMA, []),
    "fact_trips": (FACT_TRIPS_SCHEMA, ["start_year", "start_month"]),
}

TABLE_PATHS = {name: f"{WAREHOUSE}/{name}" for name in TABLES}

Path(WAREHOUSE).mkdir(parents=True, exist_ok=True)

for name, (schema, partitions) in TABLES.items():
    print(
        f"{name:12s} cols={len(schema):2d} "
        f"partitions={partitions or '-'} -> {TABLE_PATHS[name]}"
    )


stage_trips  cols=13 partitions=['source_year'] -> /opt/data/warehouse/stage_trips
dim_station  cols= 2 partitions=- -> /opt/data/warehouse/dim_station
dim_user     cols= 2 partitions=- -> /opt/data/warehouse/dim_user
fact_trips   cols=17 partitions=['start_year', 'start_month'] -> /opt/data/warehouse/fact_trips


In [6]:
# confirm warehouse creation
print(f"warehouse root : {WAREHOUSE}")
print(f"exists         : {Path(WAREHOUSE).is_dir()}\n")

for name, (schema, partitions) in TABLES.items():
    path = Path(TABLE_PATHS[name])
    # a Parquet table exists once the ETL has written data files to its path
    data_files = sorted(path.glob("**/*.parquet")) if path.is_dir() else []
    status = f"loaded ({len(data_files)} files)" if data_files else "pending"

    print(f"{name:12s} {status}")
    print(f"  path       : {path}")
    print(f"  columns    : {len(schema)}")
    print(f"  partitions : {', '.join(partitions) or '-'}")

    # partition keys must be real fields, or the ETL write will fail
    missing = [c for c in partitions if c not in schema.fieldNames()]
    if missing:
        print(f"  WARNING    : partition columns not in schema: {missing}")

    if data_files:
        df = spark.read.parquet(str(path))
        print(f"  rows       : {df.count():,}")
        if sorted(df.columns) != sorted(schema.fieldNames()):
            print("  WARNING    : on-disk columns differ from the declared schema")
    print()


warehouse root : /opt/data/warehouse
exists         : True

stage_trips  pending
  path       : /opt/data/warehouse/stage_trips
  columns    : 13
  partitions : source_year

dim_station  pending
  path       : /opt/data/warehouse/dim_station
  columns    : 2
  partitions : -

dim_user     pending
  path       : /opt/data/warehouse/dim_user
  columns    : 2
  partitions : -

fact_trips   pending
  path       : /opt/data/warehouse/fact_trips
  columns    : 17
  partitions : start_year, start_month



In [ ]:
# create data warehouse